# サイレント登録者分析

`subscriber_analytics/` の3ステージ収集パイプラインを、`youtube_network_analysis.ipynb` と同じように
セル単位で実行するノートブック。処理本体は既存スクリプト
（`collect_subscribers.py` / `collect_comments.py` / `extract_silent.py`）の関数をそのまま呼び出すため、
CLI 実行と結果は同一で、どちらから実行してもよい（データは共有される）。

## 進行状況

- [ ] Stage 1: 登録者スナップショット収集（OAuth 必須）
- [ ] Stage 2: 全動画のコメント収集（APIキー or OAuth）
- [ ] Stage 3: サイレント登録者の抽出（ローカルのみ・クォータ消費ゼロ）

## 前提条件

- `pip install -r subscriber_analytics/requirements.txt` 済み
- Stage 1: `YOUTUBE_CHANNEL_ID` と OAuth token を設定済み（初回は README の preflight を実行）
- Codespaces: `YOUTUBE_OAUTH_TOKEN_JSON` をユーザー secret として設定済み
- Stage 2: OAuth token があれば API key は不要

セットアップ手順の詳細は `subscriber_analytics/README.md` を参照。

## 共通設定

`subscriber_analytics/` のモジュールを読み込む。認証はステージごとに行う:

- **Stage 1（OAuth）**: 取得前にローカルで `preflight.py --online --authorize` を実行する。
  Codespaces では生成された token JSON を `YOUTUBE_OAUTH_TOKEN_JSON` secret から読む。
- **Stage 2（APIキー or OAuth）**: 環境変数 `YOUTUBE_API_KEY`（`.env` 可）を使う。
  Stage 1 のトークンがあれば自動で OAuth 収集に切り替わる。

In [ ]:
# 共通設定: subscriber_analytics のモジュールを読み込む
import os
import sys
from pathlib import Path

# リポジトリルートから開いても subscriber_analytics/ 内から開いても動くようにする
BASE_DIR = (
    Path("subscriber_analytics").resolve()
    if Path("subscriber_analytics").exists()
    else Path.cwd()
)
if not (BASE_DIR / "common.py").exists():
    raise RuntimeError(f"subscriber_analytics フォルダが見つかりません: {BASE_DIR}")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import common
import collect_subscribers
import collect_comments
import extract_silent

DATA_DIR = common.DATA_DIR
OUTPUT_DIR = common.OUTPUT_DIR
print(f"データ保存先: {DATA_DIR}")
print(f"抽出結果の出力先: {OUTPUT_DIR}")

## Stage 1: 登録者スナップショット収集（OAuth 必須）

`subscriptions.list`（`myRecentSubscribers=true` + `mySubscribers=true`）で自分のチャンネルの
**公開登録者**を全ページ取得し、次の2層に保存する。

- `data/snapshots/subscribers_<UTC時刻>.csv` … 実行ごとの生スナップショット
- `data/subscriber_registry.csv` … `first_seen_at` / `last_seen_at` を持つ累積レジストリ

注意:

- 取得できるのは登録を公開しているユーザーのみ（結果は常に下限値）。
- API の返却は約1,000件が上限。**週1などの定期実行**でスナップショットを積むほど
  レジストリが上限を超えて育ち、`first_seen_at` が登録日の観測ベースの近似になる。
- 自チャンネルIDが `data/channel_id.txt` に保存され、Stage 2 が既定値として利用する。

In [ ]:
# Stage 1: 登録者スナップショット収集（README の preflight 合格後に実行）
expected_channel_id = os.environ.get("YOUTUBE_CHANNEL_ID", "").strip()
if not expected_channel_id:
    raise RuntimeError("誤取得防止のため YOUTUBE_CHANNEL_ID を設定してください。")

youtube_oauth = common.build_oauth_client(
    client_secret=common.DEFAULT_CLIENT_SECRET,
    token_file=common.token_path(DATA_DIR),
)

channel_id = collect_subscribers.resolve_my_channel_id(youtube_oauth)
if channel_id != expected_channel_id:
    raise RuntimeError(f"対象チャンネル不一致: expected={expected_channel_id}, OAuth={channel_id}")
common.atomic_write_text(channel_id + "\n", common.channel_id_path(DATA_DIR))
print(f"自チャンネルID: {channel_id}（Stage 2 が既定値として利用します）")

stats = collect_subscribers.run(youtube_oauth, DATA_DIR)
print(f"スナップショット保存: {stats['snapshot_file']}")
print(
    f"取得 {stats['fetched']} 人（新規 {stats['new']} 人）/ "
    f"レジストリ累計 {stats['registry_total']} 人 / "
    f"クォータ消費 約{stats['pages'] + 1} unit"
)

## Stage 2: 全動画のコメント収集

チャンネルの uploads プレイリストから全動画を列挙し、動画ごとにコメント（返信込み・
本文は保存しない）を `data/comments/<video_id>.csv` へ保存する。

- **動画単位キャッシュ**: 保存済みの動画はスキップする（`FORCE = True` で再取得）。
  クォータ上限に達しても、翌日そのまま再実行すれば続きから収集できる。
- **`MAX_VIDEOS`**: 1回の実行で新規収集する動画数の上限（`0` = 無制限）。クォータ
  （既定 10,000 unit/日）を複数日に分割したい場合に指定する。キャッシュ済みは数えない
  ため、同じ値のまま再実行すれば続きから進む。
- **認証**: Stage 1 の OAuth トークンがあれば自動でオーナー権限収集（非公開・限定公開の
  動画も対象）。無ければ APIキー（環境変数 `YOUTUBE_API_KEY`）で**公開動画のみ**収集する。
- チャンネルIDは `CHANNEL_ID = None` のままなら Stage 1 が保存した `data/channel_id.txt`
  （または環境変数 `YOUTUBE_CHANNEL_ID`）から解決される。

In [ ]:
# Stage 2: 全動画のコメント収集（キャッシュ済みはスキップ）
CHANNEL_ID = None     # None なら data/channel_id.txt / 環境変数 YOUTUBE_CHANNEL_ID から解決
API_KEY = None        # None なら環境変数 YOUTUBE_API_KEY（.env 可）
USE_OAUTH = False     # True で常に OAuth 収集（非公開・限定公開の動画も対象）
FORCE = False         # True でキャッシュ済みの動画も再取得
MAX_VIDEOS = 0        # 1回の実行で新規収集する動画数の上限（0=無制限）

target_channel_id = collect_comments.resolve_channel_id(CHANNEL_ID, DATA_DIR)
coverage_scope = collect_comments.resolve_coverage_scope(USE_OAUTH, API_KEY, DATA_DIR)
youtube_comments = collect_comments.build_client(USE_OAUTH, API_KEY, DATA_DIR)
stats = collect_comments.run(
    youtube_comments,
    target_channel_id,
    DATA_DIR,
    force=FORCE,
    max_videos=MAX_VIDEOS,
    coverage_scope=coverage_scope,
)

print(
    f"動画 {stats['videos']} 本: 取得 {stats['collected']}（うちコメント無効 {stats['disabled']}）/ "
    f"キャッシュ済みスキップ {stats['skipped']} / 収集コメント {stats['comments']} 件 / "
    f"クォータ消費 約{stats['pages']} unit"
)
print(f"収集状態: {stats['videos_cached']}/{stats['videos']} 本 → {stats['state_file']}")
if not stats["comment_coverage_complete"]:
    print("未取得動画があります。同じ MAX_VIDEOS で再実行すると続きから収集します。")
if stats["skipped"] and not stats["collected"]:
    print("すべてキャッシュ済みです。再取得する場合は FORCE = True にしてください。")

## Stage 3: サイレント登録者の抽出（ローカルのみ・クォータ消費ゼロ）

Stage 1 のレジストリと Stage 2 のコメントキャッシュを突合し、期間条件でフィルタした
登録者一覧を `output/silent_subscribers_<日付>.csv` に出力する。API は呼ばないため、
**条件を変えて何度でも再実行できる**。

### フィルタ設定（組み合わせは AND）

| 変数 | 例 | 意味 |
|---|---|---|
| `SUBSCRIBED_WITHIN` | `"90d"` | 登録が直近この期間以内（`12w` / `3m` も可） |
| `SUBSCRIBED_SINCE` / `SUBSCRIBED_UNTIL` | `"2026-04-01"` | 登録日の下限 / 上限（YYYY-MM-DD、両端含む） |
| `NEVER_COMMENTED` | `True` | 一度もコメントしていない人のみ |
| `NO_COMMENT_WITHIN` | `"90d"` | 直近この期間コメントしていない人（過去のコメント有無は不問） |
| `INCLUDE_UNSUBSCRIBED` | `True` | 最新スナップショットに出現しなかった人（解約の可能性）も含める |
| `ALLOW_INCOMPLETE_COMMENTS` | `True` | 未完了コメントでも続行（偽陽性の可能性があるため非推奨） |

すべて既定値（フィルタなし）のまま実行すると、全登録者を4象限セグメント付きで出力する。
セグメント: **新規サイレント**（期間内登録・コメント0）/ **古参サイレント**（それ以前の
登録・コメント0）/ **休眠**（過去コメントあり・期間内なし）/ **アクティブ**。

例: 「直近3ヶ月以内に登録したが一度もコメントしていない人」は
`SUBSCRIBED_WITHIN = "90d"` と `NEVER_COMMENTED = True` を設定して実行する。

In [ ]:
# Stage 3: サイレント登録者の抽出（フィルタはこのセル冒頭の変数で指定）
from types import SimpleNamespace

SUBSCRIBED_WITHIN = None      # 例: "90d"（登録が直近この期間以内）
SUBSCRIBED_SINCE = None       # 例: "2026-04-01"（登録日の下限）
SUBSCRIBED_UNTIL = None       # 例: "2026-06-30"（登録日の上限）
NEVER_COMMENTED = False       # True: 一度もコメントしていない人のみ
NO_COMMENT_WITHIN = None      # 例: "90d"（直近この期間コメントしていない人）
INCLUDE_UNSUBSCRIBED = False  # True: 解約した可能性のある人も含める
ALLOW_INCOMPLETE_COMMENTS = False  # True: 未完了コメントでも続行（非推奨）
OUT_PATH = None               # 出力CSVパス（None なら output/silent_subscribers_<日付>.csv）

args = SimpleNamespace(
    subscribed_within=SUBSCRIBED_WITHIN,
    subscribed_since=SUBSCRIBED_SINCE,
    subscribed_until=SUBSCRIBED_UNTIL,
    never_commented=NEVER_COMMENTED,
    no_comment_within=NO_COMMENT_WITHIN,
)
now = common.utcnow()

comment_state = extract_silent.validate_comment_collection(
    DATA_DIR, allow_incomplete=ALLOW_INCOMPLETE_COMMENTS
)
registry = extract_silent.load_registry(DATA_DIR)

# 既定では最新スナップショットに出現した人（現役の公開登録者）のみを対象にする。
# レジストリには解約済みの人も last_seen_at が古いまま残るため。
excluded_unsubscribed = 0
latest_seen = registry["last_seen_at"].max() if len(registry) else ""
if not INCLUDE_UNSUBSCRIBED and latest_seen:
    current = registry["last_seen_at"] == latest_seen
    excluded_unsubscribed = int((~current).sum())
    registry = registry[current]

comment_stats, n_comment_files = extract_silent.load_comment_stats(DATA_DIR)
if n_comment_files == 0 and int(comment_state.get("videos_listed", 0)) > 0:
    raise RuntimeError("動画が存在しますがコメントキャッシュがありません。Stage 2 を再実行してください。")

table = extract_silent.build_table(registry, comment_stats)
recent_window = common.parse_duration(SUBSCRIBED_WITHIN or "90d")
activity_window = common.parse_duration(NO_COMMENT_WITHIN or "90d")
table = extract_silent.add_segments(table, now, recent_window, activity_window)

filtered, applied = extract_silent.apply_filters(table, args, now)
result = extract_silent.format_output(filtered)
out_path = Path(OUT_PATH) if OUT_PATH else (
    OUTPUT_DIR / f"silent_subscribers_{now.strftime('%Y%m%d')}.csv"
)
common.atomic_write_csv(result, out_path)

scope = "現役登録者" if not INCLUDE_UNSUBSCRIBED else "レジストリ全体"
print(f"=== セグメント集計（{scope} {len(table)} 人 / 基準時刻 {common.format_ts(now)}） ===")
if excluded_unsubscribed:
    print(
        f"  ※最新スナップショット（{latest_seen}）に出現しなかった {excluded_unsubscribed} 人"
        "（解約の可能性）を除外済み。含めるには INCLUDE_UNSUBSCRIBED = True"
    )
for name in (
    extract_silent.SEG_NEW_SILENT,
    extract_silent.SEG_OLD_SILENT,
    extract_silent.SEG_DORMANT,
    extract_silent.SEG_ACTIVE,
):
    print(f"  {name}: {int((table['segment'] == name).sum())} 人")
if applied:
    print("適用フィルタ: " + " AND ".join(applied))
else:
    print("適用フィルタ: なし（全登録者をセグメント付きで出力）")
print(f"出力: {len(filtered)} 人 → {out_path}")
print("注記: 対象は登録を公開しているユーザーのみのため、実際の人数の下限値です。")

result.head(20)